# Análise de dados Industriais

#### Importações

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime


In [2]:
# Função para detectar outliers
def listar_outliers(df, coluna):
    """
    Retorna um novo DataFrame contendo apenas os outliers da coluna especificada.
    Utiliza o método do Intervalo Interquartil (IQR).
    """
    # 1. Calcular os quartis e o IQR
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1

    # 2. Definir os limites inferior e superior
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # 3. Filtrar o DataFrame para retornar apenas os outliers
    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

    return outliers

In [44]:
cod_maquina = {
    'MAQ-01': 'TORNO CNC 01',
    'MAQ-02': 'TORNO CNC 02',
    'MAQ-03': 'FRESADORA 01',
    'MAQ-04': 'FRESADORA 02',
    'MAQ-05': 'PRENSA HIDRÁULICA 01',
    'MAQ-06': 'PRENSA HIDRÁULICA 02',
    'MAQ-07': 'SOLDA ROBOTIZADA 01',
    'MAQ-08': 'SOLDA ROBOTIZADA 02',
    'MAQ-09': 'INJETORA PLÁSTICA 01',
    'MAQ-10': 'EXTRUSORA 01',
    'MAQ-11': 'INDETERMINADO'
}

### Extração e Tratamento dos dados sensores.json

In [46]:
df_sensores = pd.read_json('/content/sensores.json')
df_medicoes = pd.DataFrame(dict(df_sensores['medicoes'])).T

df_sensores = df_sensores.drop(columns=['medicoes'], axis=1)

df_sensores = pd.concat([df_sensores, df_medicoes], axis=1)

# O df_sensores apresentou inconsistencia na quantidade de registro da serie 'temperaturas'
# Etapa 1 verificar a quantidade de registos nulos
# print(df_sensores.isna().sum())
# Verificado 6 registros nulos
# Etapa 2 substituir os valores nulos pela mediana
df_sensores['temperatura'] = df_sensores['temperatura'].fillna(df_sensores['temperatura'].median())

# Verificar duplicados
# print(f'Duplicados: {df_sensores.duplicated().sum()}')
# verificado 4 duplicados
# Etapa 3 excluir duplicados
df_sensores = df_sensores.drop_duplicates()
# converter para datetime e padronizar o formato para futuro merge
df_sensores['timestamp'] = pd.to_datetime(df_sensores['timestamp']).dt.strftime('%Y-%m-%d')
# renomear a coluna timestamp para data
df_sensores = df_sensores.rename(columns={'timestamp': 'data'})
df_sensores['data'] = pd.to_datetime(df_sensores['data'])

df_sensores.insert(1, 'maquina', df_sensores['id_maquina'].map(cod_maquina))

df_sensores




,id_maquina,maquina,data,status,consumo_energia,temperatura,vibracao,pressao,velocidade
0,MAQ-03,FRESADORA 01,2025-03-20,operando,11.86,68.1,2.32,5.93,1288.8
1,MAQ-10,EXTRUSORA 01,2025-03-24,operando,13.89,60.4,1.91,4.49,1020.1
2,MAQ-10,EXTRUSORA 01,2025-03-23,operando,9.88,65.5,2.08,4.24,1181.3
3,MAQ-02,TORNO CNC 02,2025-03-04,operando,13.63,66.4,2.36,5.34,1153.6
4,MAQ-07,SOLDA ROBOTIZADA 01,2025-03-24,operando,14.98,70.5,2.77,5.73,1155.1
...,...,...,...,...,...,...,...,...,...
609,MAQ-02,TORNO CNC 02,2025-03-09,operando,11.03,68.3,2.20,5.66,1289.3
610,MAQ-03,FRESADORA 01,2025-03-27,operando,15.25,70.1,1.88,5.68,1158.5
611,MAQ-07,SOLDA ROBOTIZADA 01,2025-03-19,operando,10.36,64.8,1.89,6.01,1276.7
612,MAQ-08,SOLDA ROBOTIZADA 02,2025-03-21,operando,13.10,65.5,3.20,4.43,1169.6


### Extracao e Tratamento dos dados producao.csv

In [42]:
df_producao = pd.read_csv('/content/producao.csv')

# Etapa 1 Verificar dados nulos
# print(df_producao.isna().sum())
# identificado valores nulos na 'quantidade_produzida', 'tempo_producao' e 'operador'
# Substituir valores nulos de 'quantidade_produzida', 'tempo_producao' pela mediana
df_producao['quantidade_produzida'] = df_producao['quantidade_produzida']\
                .fillna(df_producao['quantidade_produzida'].median())
df_producao['tempo_producao'] = df_producao['tempo_producao']\
                .fillna(df_producao['tempo_producao'].median())
# Substituir valores nulos de 'operador' pela mediana
df_producao['operador'] = df_producao['operador']\
                .fillna(df_producao['operador'].mode()[0])

# Etapa 2 Verificar Duplicados
# print(df_producao.duplicated().sum())
# Localizado 5 valores duplciados
df_producao = df_producao.drop_duplicates()
# Convertendo data para datetime e padronizando o formato
df_producao['data'] = pd.to_datetime(df_producao['data'], yearfirst=True, format='mixed')

# Etapa 3 Padronização dos nomes das Máquinas
# print(df_producao['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_producao['maquina'] = df_producao['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_producao['maquina'] = df_producao['maquina'].str[:-1] + '0' + df_producao['maquina'].str[-1]
df_producao['maquina'] = df_producao['maquina'].str.replace('00', '0')

# print('\n Após padronização do nome')
# print(df_producao['maquina'].unique())

# Foi identificado valores iguais a zero em tempo_producao
# print(df_producao[df_producao['tempo_producao'] == 0][['maquina', 'tempo_producao']])
# Substitui valores 0 pela mediana do tempo_producao e arendodar para duas casas decimais
df_producao['tempo_producao'] = df_producao['tempo_producao'].replace(0, df_producao['tempo_producao'].median()).round(2)

# padronizando os turnos
# print(df_producao['turno'].unique())
df_producao['turno'] = df_producao['turno'].str.strip().str.lower()
# print(df_producao['turno'].unique())

df_producao['quantidade_produzida'] = df_producao['quantidade_produzida'].astype('int')


# listar_outliers(df_producao, 'tempo_producao')

df_producao[df_producao['quantidade_produzida'] < 0]

listar_outliers(df_producao, 'tempo_producao')










,id_producao,data,maquina,produto,quantidade_produzida,quantidade_planejada,tempo_producao,operador,turno
35,PR0167,2025-03-14,EXTRUSORA 01,Suporte Metálico,560,600,2226.28,Fernanda Lima,tarde
41,PR0326,2025-03-29,SOLDA ROBOTIZADA 02,Engrenagem Industrial,308,350,552.00,Marcos Pereira,noite
189,PR0029,2025-03-03,SOLDA ROBOTIZADA 02,Carcaça Plástica,615,700,2158.82,Fernanda Lima,manhã
346,PR0057,2025-05-03,SOLDA ROBOTIZADA 02,Suporte Metálico,560,600,2704.66,Fernanda Lima,noite


### Extracao e Tratamento dos dados manutencao.xlsx

In [21]:
df_manutencao = pd.read_excel('manutencao.xlsx')

# df_manutencao.isna().sum()
# Valores Nulos pela mediana
df_manutencao['tempo_parada'] = df_manutencao['tempo_parada'].fillna(df_manutencao['tempo_parada'].median())
df_manutencao['tempo_parada'] = df_manutencao['tempo_parada'].astype('int')
# df_manutencao.isna().sum()

# Verifica duplicados
# df_manutencao.duplicated().sum()

# Padronização Nomes das Maquinas
# print(df_manutencao['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_manutencao['maquina'] = df_manutencao['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_manutencao['maquina'] = df_manutencao['maquina'].str[:-1] + '0' + df_manutencao['maquina'].str[-1]
df_manutencao['maquina'] = df_manutencao['maquina'].str.replace('00', '0')
# print(df_manutencao['maquina'].unique())

# Padronizando data
df_manutencao['data'] = pd.to_datetime(df_manutencao['data'], format='mixed', yearfirst=True)

# Padronização dos tipos
# print(df_manutencao['tipo_manutencao'].unique())
df_manutencao['tipo_manutencao'] = df_manutencao['tipo_manutencao'].str.strip().str.upper().str.replace('  ', ' ')
# print(df_manutencao['tipo_manutencao'].unique())

df_manutencao.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_manutencao     93 non-null     object        
 1   maquina           93 non-null     object        
 2   data              93 non-null     datetime64[ns]
 3   tipo_manutencao   93 non-null     object        
 4   motivo            93 non-null     object        
 5   tempo_parada      93 non-null     int64         
 6   custo_manutencao  93 non-null     float64       
 7   tecnico           93 non-null     object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 5.9+ KB


### Extracao e Tratamento dos dados qualidade.parquet

In [39]:
df_qualidade = pd.read_parquet('/content/qualidade.parquet')

# Verificando e atualizando valores nulos para a mediana
df_qualidade.isna().sum()
df_qualidade['quantidade_aprovada'] = df_qualidade['quantidade_aprovada'].fillna(df_qualidade['quantidade_aprovada'].median())

# Verificando e deletando duplicados
# df_qualidade.duplicated().sum()

# Padronização Nomes das Maquinas
# print(df_qualidade['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_qualidade['maquina'] = df_qualidade['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_qualidade['maquina'] = df_qualidade['maquina'].str[:-1] + '0' + df_qualidade['maquina'].str[-1]
df_qualidade['maquina'] = df_qualidade['maquina'].str.replace('00', '0')
# print(df_qualidade['maquina'].unique())

df_qualidade['quantidade_aprovada'] = df_qualidade['quantidade_aprovada'].astype('int')
df_qualidade['meta_qualidade'] = df_qualidade['meta_qualidade'].astype('int')

# Visualisar valores acima de 100 do indice_qualidade e substituir pela mediana
# print(df_qualidade[df_qualidade['indice_qualidade'] > 100])
# Calculo da mediana sobre os indices <= 100
mediana_iq = df_qualidade[df_qualidade['indice_qualidade'] <= 100]['indice_qualidade'].median()
# Substituição dos valores inconsistentes pela mediana
df_qualidade.loc[df_qualidade['indice_qualidade'] > 100, 'indice_qualidade'] = mediana_iq



Empty DataFrame
Columns: [id_producao, maquina, produto, quantidade_inspecionada, quantidade_aprovada, quantidade_rejeitada, indice_qualidade, meta_qualidade]
Index: []
